# 014 - QAOA via simulator (V 1.0)  

### Admin

In [1]:
# Data Utilites 
import numpy as np

# Evironmental Libraries 
from tabulate import tabulate
import environment as envir

from qiskit_algorithms.utils import algorithm_globals
from qiskit_algorithms import SamplingVQE
from qiskit_algorithms.optimizers import COBYLA

from qiskit.primitives import StatevectorSampler
from qiskit.circuit.library import n_local

from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_finance.applications.optimization import PortfolioOptimization

In [2]:
data, headers = envir.environment_state()
print(tabulate(data, headers=headers))

System/Library           Version
-----------------------  ---------
Python                   3.11.15
numpy                    2.4.3
torch                    2.10.0
matplotlib               3.10.8
seaborn                  0.13.2
qiskit                   2.2.0
qiskit_machine_learning  0.9.0
qiskit_optimization      0.7.0
qiskit_algorithms        0.4.0
yfinance                 1.2.0
qiskit_aer               0.15.1
qiskit-finance           0.4.1


### Notebook Parameters 

In [3]:
# Set Seed 
algorithm_globals.random_seed = 1234

# Print (debug) 
print_flag = True 

### The Data 

In [4]:
num_assets = 4 

In [5]:
expected_returns = np.array([0.10, 0.20, 0.15, 0.18])

covariances = np.array([
    [0.005, -0.010, 0.004, -0.002],
    [-0.010, 0.040, -0.002, 0.004],
    [0.004, -0.002, 0.023, 0.002],
    [-0.002, 0.004, 0.002, 0.018],
])

risk_factor = 0.5
budget = 2

### The Model

In [6]:
portfolio = PortfolioOptimization(
    expected_returns,
    covariances,
    risk_factor,
    budget
)

qp = portfolio.to_quadratic_program()

if print_flag: 
    print(qp.prettyprint())

Problem name: Portfolio optimization

Minimize
  0.0025*x_0^2 - 0.01*x_0*x_1 + 0.004*x_0*x_2 - 0.002*x_0*x_3 + 0.02*x_1^2
  - 0.002*x_1*x_2 + 0.004*x_1*x_3 + 0.0115*x_2^2 + 0.002*x_2*x_3 + 0.009*x_3^2
  - 0.1*x_0 - 0.2*x_1 - 0.15*x_2 - 0.18*x_3

Subject to
  Linear constraints (1)
    x_0 + x_1 + x_2 + x_3 == 2  'c0'

  Binary variables (4)
    x_0 x_1 x_2 x_3



In [7]:
ansatz = n_local(
    num_qubits=num_assets,
    rotation_blocks=["ry"],
    entanglement_blocks="cz",
    entanglement="full",
    reps=3
)

In [8]:
# Classical Optimizer 
optimizer = COBYLA(maxiter=500)

In [9]:
# Create the Sampler 
sampler = StatevectorSampler()

In [10]:
vqe = SamplingVQE(
    sampler=sampler,
    ansatz=ansatz,
    optimizer=optimizer
)

In [11]:
solver = MinimumEigenOptimizer(vqe)

### Solve 

In [12]:
result = solver.solve(qp)

### Post Processing 

In [13]:
print("Optimal portfolio:", result.x)
print("Optimal value:", result.fval)

Optimal portfolio: [0. 1. 0. 1.]
Optimal value: -0.347


In [14]:
print("Objective value from qp:", qp.objective.evaluate(result.x))

Objective value from qp: -0.347
